In [1]:
import pandas as pd
import os

# Cek versi — pastikan pandas terbaca
print("Pandas version:", pd.__version__)

Pandas version: 3.0.3


In [2]:
import pandas as pd
import os

RAW_PATH = "../data/raw/"

files = os.listdir(RAW_PATH)
for f in files:
    print(f)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [3]:
customers   = pd.read_csv(RAW_PATH + "olist_customers_dataset.csv")
orders      = pd.read_csv(RAW_PATH + "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_PATH + "olist_order_items_dataset.csv")
payments    = pd.read_csv(RAW_PATH + "olist_order_payments_dataset.csv")
reviews     = pd.read_csv(RAW_PATH + "olist_order_reviews_dataset.csv")
products    = pd.read_csv(RAW_PATH + "olist_products_dataset.csv")
sellers     = pd.read_csv(RAW_PATH + "olist_sellers_dataset.csv")
geolocation = pd.read_csv(RAW_PATH + "olist_geolocation_dataset.csv")
category    = pd.read_csv(RAW_PATH + "product_category_name_translation.csv")

print("✅ Semua tabel berhasil di-load")

✅ Semua tabel berhasil di-load


In [4]:
tables = {
    "customers":   customers,
    "orders":      orders,
    "order_items": order_items,
    "payments":    payments,
    "reviews":     reviews,
    "products":    products,
    "sellers":     sellers,
    "geolocation": geolocation,
    "category":    category
}

for name, df in tables.items():
    print(f"{name:15s} → {df.shape[0]:>7,} rows  x  {df.shape[1]:>2} cols")

customers       →  99,441 rows  x   5 cols
orders          →  99,441 rows  x   8 cols
order_items     → 112,650 rows  x   7 cols
payments        → 103,886 rows  x   5 cols
reviews         →  99,224 rows  x   7 cols
products        →  32,951 rows  x   9 cols
sellers         →   3,095 rows  x   4 cols
geolocation     → 1,000,163 rows  x   5 cols
category        →      71 rows  x   2 cols


## Observasi Awal Ukuran Data

- `orders` dan `customers` sama-sama 99,441 baris → relasi 1-to-1 per customer
- `order_items` punya 112,650 baris > orders → 1 order bisa punya beberapa item (normal)
- `payments` punya 103,886 baris > orders → ada order yang dibayar lebih dari 1 metode (misalnya kartu kredit + voucher)
- `reviews` hanya 99,224 baris → ada ~217 order tanpa review, perlu diperhatikan saat merge
- `geolocation` sangat besar (1 juta+ baris) → tidak akan masuk ke model utama, di-skip untuk sekarang

In [5]:
# Lihat kolom dan sample data setiap tabel
for name, df in tables.items():
    if name == "geolocation":
        continue  # skip, terlalu besar
    print(f"\n{'='*50}")
    print(f"  {name.upper()}")
    print(f"{'='*50}")
    print(df.dtypes)
    print(f"\nSample:")
    print(df.head(2))


  CUSTOMERS
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Sample:
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  

  ORDERS
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

Sample:
                           order_id      

## Merge Tabel
Strategi merge bertahap:

1. `orders` + `customers` → tambahkan info customer ke setiap order
2. `order_items` → tambahkan detail item yang dibeli
3. `payments` → tambahkan info pembayaran
4. `reviews` → tambahkan rating customer
5. `products` + `category` → tambahkan nama kategori produk

Menggunakan **LEFT JOIN** agar semua order tetap ada meskipun data pasangannya tidak lengkap.

In [6]:
# Step 1: orders + customers
df = orders.merge(customers, on="customer_id", how="left")
print(f"Step 1 - orders + customers: {df.shape}")

# Step 2: + order_items
df = df.merge(order_items, on="order_id", how="left")
print(f"Step 2 - + order_items:      {df.shape}")

# Step 3: + payments
payments_agg = payments.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    payment_installments=("payment_installments", "max"),
    payment_type=("payment_type", "first")
).reset_index()

df = df.merge(payments_agg, on="order_id", how="left")
print(f"Step 3 - + payments:         {df.shape}")

# Step 4: + reviews
reviews_agg = reviews.groupby("order_id").agg(
    review_score=("review_score", "mean")
).reset_index()

df = df.merge(reviews_agg, on="order_id", how="left")
print(f"Step 4 - + reviews:          {df.shape}")

# Step 5: + products + category
products_cat = products.merge(category, on="product_category_name", how="left")
df = df.merge(
    products_cat[["product_id", "product_category_name_english"]],
    on="product_id", how="left"
)
print(f"Step 5 - + products+category: {df.shape}")

print("\n✅ Master dataframe selesai dibuat")
print(f"Final shape: {df.shape}")

Step 1 - orders + customers: (99441, 12)
Step 2 - + order_items:      (113425, 18)
Step 3 - + payments:         (113425, 21)
Step 4 - + reviews:          (113425, 22)
Step 5 - + products+category: (113425, 23)

✅ Master dataframe selesai dibuat
Final shape: (113425, 23)


## Hasil Merge

- Final shape: 113,425 baris x 23 kolom
- Baris bertambah dari 99,441 (orders) → 113,425 karena 1 order bisa punya beberapa item
- Setiap baris sekarang merepresentasikan 1 item dalam 1 order, bukan 1 order
- Semua info customer, payment, review, dan kategori produk sudah tergabung

In [7]:
# Lihat kolom apa saja yang ada
print("Kolom di master dataframe:")
print(df.columns.tolist())

# Cek sekilas
df.head(3)

Kolom di master dataframe:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'payment_value', 'payment_installments', 'payment_type', 'review_score', 'product_category_name_english']


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,product_id,seller_id,shipping_limit_date,price,freight_value,payment_value,payment_installments,payment_type,review_score,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,38.71,1.0,credit_card,4.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,141.46,1.0,boleto,4.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,179.12,3.0,credit_card,5.0,auto


In [8]:
# VALIDASI POST-MERGE
# Memastikan hasil merge bersih sebelum disimpan
print("=" * 50)
print("  VALIDASI POST-MERGE")
print("=" * 50)

# 1. Cek duplikat
n_duplicates = df.duplicated().sum()
print(f"\n[1] Duplikat rows    : {n_duplicates}")
if n_duplicates > 0:
    print("    ⚠️  Ada duplikat — perlu investigasi")
else:
    print("    ✅ Tidak ada duplikat")

# 2. Cek row count masuk akal
# Setelah merge order_items, row harusnya > orders (1 order bisa punya banyak item)
print(f"\n[2] Row count check:")
print(f"    orders original  : {orders.shape[0]:>8,}")
print(f"    master (final)   : {df.shape[0]:>8,}")
print(f"    selisih          : +{df.shape[0] - orders.shape[0]:>7,} (dari multi-item orders)")

# 3. Missing values per kolom kunci
key_cols = ['customer_unique_id', 'order_id', 'order_purchase_timestamp',
            'payment_value', 'review_score']
print(f"\n[3] Missing values di kolom kunci:")
for col in key_cols:
    if col in df.columns:
        n_miss = df[col].isnull().sum()
        pct    = n_miss / len(df) * 100
        flag   = "⚠️" if pct > 5 else "✅"
        print(f"    {flag} {col:<35}: {n_miss:>6,} missing ({pct:.1f}%)")

# 4. Pastikan tidak ada order_id yang hilang setelah merge
orders_in_master = df['order_id'].nunique()
print(f"\n[4] Order ID coverage:")
print(f"    Unique orders di orders.csv : {orders['order_id'].nunique():>7,}")
print(f"    Unique orders di master     : {orders_in_master:>7,}")
if orders_in_master == orders['order_id'].nunique():
    print("    ✅ Semua order terwakili")
else:
    print(f"    ⚠️  Ada {orders['order_id'].nunique() - orders_in_master} order yang hilang!")

print(f"\n✅ Validasi selesai — master.csv siap disimpan")

  VALIDASI POST-MERGE



[1] Duplikat rows    : 0
    ✅ Tidak ada duplikat

[2] Row count check:
    orders original  :   99,441
    master (final)   :  113,425
    selisih          : + 13,984 (dari multi-item orders)

[3] Missing values di kolom kunci:
    ✅ customer_unique_id                 :      0 missing (0.0%)
    ✅ order_id                           :      0 missing (0.0%)
    ✅ order_purchase_timestamp           :      0 missing (0.0%)
    ✅ payment_value                      :      3 missing (0.0%)
    ✅ review_score                       :    961 missing (0.8%)

[4] Order ID coverage:
    Unique orders di orders.csv :  99,441
    Unique orders di master     :  99,441
    ✅ Semua order terwakili

✅ Validasi selesai — master.csv siap disimpan


## Hasil Validasi Post-Merge

Hasil pengecekan sebelum master.csv disimpan:

- **Duplikat**: 0 — data bersih, tidak ada baris ganda
- **Row count**: master (113,425) > orders (99,441) karena 1 order bisa punya beberapa item — selisih 13,984 baris adalah perilaku yang diharapkan dari merge dengan order_items
- **Missing values di kolom kunci**: hanya `payment_value` (3 rows / 0.0%) dan `review_score` (961 rows / 0.8%) — keduanya akan di-handle di tahap Feature Engineering
- **Order ID coverage**: semua 99,441 order dari orders.csv terwakili di master — tidak ada data yang hilang saat merge

In [9]:
# Simpan master dataframe
df.to_csv("../data/processed/master.csv", index=False)

print("✅ File disimpan ke data/processed/master.csv")
print(f"Ukuran file: {df.shape}")

✅ File disimpan ke data/processed/master.csv
Ukuran file: (113425, 23)


## Summary Notebook 01 - Data Loading

Dataset: Olist Brazilian E-Commerce (Kaggle)

Tabel yang diload: customers, orders, order_items, payments, reviews, products, sellers, category (8 tabel aktif, geolocation di-skip)

Proses:
- Load 9 CSV ke dataframe terpisah
- Analisis ukuran dan struktur setiap tabel
- Merge bertahap menggunakan LEFT JOIN
- Validasi post-merge: 0 duplikat, semua 99,441 order terwakili, missing values hanya di `payment_value` (3 rows) dan `review_score` (961 rows / 0.8%)
- Output: master.csv (113,425 baris x 23 kolom) disimpan ke data/processed/

Next: EDA di notebook 02_eda.ipynb